ilk basda python ve sqli ise salaq

In [18]:
import sys
from pathlib import Path
import pandas as pd

from sqlalchemy import text, Unicode

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src" / "db.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
from src.db import get_engine

engine = get_engine()

print("Python və SQL Server hazırdır!")

Python və SQL Server hazırdır!


csv fayllarimizi python data frame formasina getirek

In [19]:
DATA_DIR = PROJECT_ROOT / "data" / "bank_data"

customers = pd.read_csv(DATA_DIR / "customers.csv")
loans = pd.read_csv(DATA_DIR / "loans.csv")
transactions = pd.read_csv(DATA_DIR / "transactions.csv")

print("Customers:", customers.shape)
print("Loans:", loans.shape)
print("Transactions:", transactions.shape)

Customers: (5000, 13)
Loans: (4820, 10)
Transactions: (77396, 8)


fayllari sql servere yazaq

In [20]:
customers.to_sql("customers", engine, schema="bronze", if_exists="replace",
    index=False, dtype={c: Unicode(255) for c in customers.select_dtypes(include=["object", "string"]).columns})

loans.to_sql("loans", engine, schema="bronze", if_exists="replace",
    index=False, dtype={c: Unicode(255) for c in loans.select_dtypes(include=["object", "string"]).columns})

transactions.to_sql("transactions", engine, schema="bronze", if_exists="replace",
    index=False, dtype={c: Unicode(255) for c in transactions.select_dtypes(include=["object", "string"]).columns})

106

Kredit və müştəri məlumatlarını birləşdirək

In [21]:
with engine.connect() as conn:
    customers_df = pd.read_sql(
        text("SELECT * FROM bronze.customers"), conn
    )

    loans_df = pd.read_sql(
        text("SELECT * FROM bronze.loans"), conn
    )

bronze_loans_merged = loans_df.merge(
    customers_df,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

bronze_loans_merged.head()

,loan_id,customer_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted,...,city,branch,tenure_years,credit_score,balance,salary,products_count,has_card,is_active,churned
0,L000001,C03267,1/1/2026,Biznes krediti,18174.08,34,11.34,Təsdiqləndi,NaN,0.0,...,Mingəçevir,Mingəçevir Mərkəz,5.0,682.0,15988.68,2175.84,2.0,1.0,1.0,0.0
1,L000002,C00082,1/2/2026,İstehlak krediti,14942.94,15,13.74,Rədd edildi,Yüksək borc yükü,NaN,...,Lənkəran,Lənkəran Mərkəz,4.0,624.0,30895.36,2077.34,1.0,0.0,1.0,1.0
2,L000003,C00715,1/2/2026,İstehlak krediti,24217.34,58,14.09,Təsdiqləndi,NaN,0.0,...,Bakı,Səbail,5.0,700.0,1473.43,1912.54,2.0,0.0,0.0,1.0
3,L000004,C02969,1/2/2026,Biznes krediti,14890.76,58,14.82,Rədd edildi,Aşağı kredit skoru,NaN,...,Gəncə,Gəncə Mərkəz,12.0,544.0,0.00,1020.38,1.0,0.0,1.0,1.0
4,L000005,C00178,1/3/2026,İstehlak krediti,6382.58,21,13.19,Təsdiqləndi,NaN,0.0,...,Lənkəran,Lənkəran Mərkəz,6.0,850.0,6541.33,2214.41,1.0,1.0,1.0,0.0


Bayram təqvimini kreditlərə birləşdirək

In [22]:
# Bayram təqvimini Bronze-dan oxuyuruq
with engine.connect() as conn:
    calendar_df = pd.read_sql(
        text("SELECT * FROM bronze.calendar_reference"),
        conn
    )

# Tarixləri birləşdirmə üçün eyni formata gətiririk
bronze_loans_merged["merge_date"] = pd.to_datetime(
    bronze_loans_merged["application_date"]
).dt.date

calendar_df["merge_date"] = pd.to_datetime(
    calendar_df["date"]
).dt.date

# Bayram məlumatlarını kreditlərə əlavə edirik
bronze_loans_merged = bronze_loans_merged.merge(
    calendar_df[
        ["merge_date", "is_holiday", "holiday_name",
         "is_weekend", "is_workday", "is_pre_holiday"]
    ],
    on="merge_date",
    how="left",
    validate="many_to_one"
)
bronze_loans_merged.head()


,loan_id,customer_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted,...,products_count,has_card,is_active,churned,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday
0,L000001,C03267,1/1/2026,Biznes krediti,18174.08,34,11.34,Təsdiqləndi,NaN,0.0,...,2.0,1.0,1.0,0.0,2026-01-01,1,Yeni il,0,0,0
1,L000002,C00082,1/2/2026,İstehlak krediti,14942.94,15,13.74,Rədd edildi,Yüksək borc yükü,NaN,...,1.0,0.0,1.0,1.0,2026-01-02,1,Yeni il,0,0,0
2,L000003,C00715,1/2/2026,İstehlak krediti,24217.34,58,14.09,Təsdiqləndi,NaN,0.0,...,2.0,0.0,0.0,1.0,2026-01-02,1,Yeni il,0,0,0
3,L000004,C02969,1/2/2026,Biznes krediti,14890.76,58,14.82,Rədd edildi,Aşağı kredit skoru,NaN,...,1.0,0.0,1.0,1.0,2026-01-02,1,Yeni il,0,0,0
4,L000005,C00178,1/3/2026,İstehlak krediti,6382.58,21,13.19,Təsdiqləndi,NaN,0.0,...,1.0,1.0,1.0,0.0,2026-01-03,0,NaN,1,0,0


In [23]:
bronze_loans_merged.head()

,loan_id,customer_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted,...,products_count,has_card,is_active,churned,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday
0,L000001,C03267,1/1/2026,Biznes krediti,18174.08,34,11.34,Təsdiqləndi,NaN,0.0,...,2.0,1.0,1.0,0.0,2026-01-01,1,Yeni il,0,0,0
1,L000002,C00082,1/2/2026,İstehlak krediti,14942.94,15,13.74,Rədd edildi,Yüksək borc yükü,NaN,...,1.0,0.0,1.0,1.0,2026-01-02,1,Yeni il,0,0,0
2,L000003,C00715,1/2/2026,İstehlak krediti,24217.34,58,14.09,Təsdiqləndi,NaN,0.0,...,2.0,0.0,0.0,1.0,2026-01-02,1,Yeni il,0,0,0
3,L000004,C02969,1/2/2026,Biznes krediti,14890.76,58,14.82,Rədd edildi,Aşağı kredit skoru,NaN,...,1.0,0.0,1.0,1.0,2026-01-02,1,Yeni il,0,0,0
4,L000005,C00178,1/3/2026,İstehlak krediti,6382.58,21,13.19,Təsdiqləndi,NaN,0.0,...,1.0,1.0,1.0,0.0,2026-01-03,0,NaN,1,0,0


Hava məlumatlarını birləşdirək

In [24]:
weather_df = pd.read_sql("SELECT * FROM bronze.weather", engine)
weather_df["merge_date"] = pd.to_datetime(weather_df["time"]).dt.date

In [25]:
bronze_loans_merged = bronze_loans_merged.merge(
    weather_df, on=["merge_date", "city"], how="left"
)
bronze_loans_merged.head()

,loan_id,customer_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted,...,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday,time,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,L000001,C03267,1/1/2026,Biznes krediti,18174.08,34,11.34,Təsdiqləndi,NaN,0.0,...,2026-01-01,1,Yeni il,0,0,0,2026-01-01,5.5,0.0,18.6
1,L000002,C00082,1/2/2026,İstehlak krediti,14942.94,15,13.74,Rədd edildi,Yüksək borc yükü,NaN,...,2026-01-02,1,Yeni il,0,0,0,2026-01-02,5.2,12.9,22.6
2,L000003,C00715,1/2/2026,İstehlak krediti,24217.34,58,14.09,Təsdiqləndi,NaN,0.0,...,2026-01-02,1,Yeni il,0,0,0,2026-01-02,2.9,5.3,36.6
3,L000004,C02969,1/2/2026,Biznes krediti,14890.76,58,14.82,Rədd edildi,Aşağı kredit skoru,NaN,...,2026-01-02,1,Yeni il,0,0,0,2026-01-02,2.0,0.7,25.5
4,L000005,C00178,1/3/2026,İstehlak krediti,6382.58,21,13.19,Təsdiqləndi,NaN,0.0,...,2026-01-03,0,NaN,1,0,0,2026-01-03,3.6,0.0,10.7


Birləşdirilmiş cədvəli Bronze-a yazaq

In [26]:
# Birləşdirilmiş kredit məlumatlarını Bronze-a yazırıq
bronze_loans_merged.to_sql(
    name="loans_enriched",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False, 
    dtype={c: Unicode(255) for c in bronze_loans_merged.select_dtypes(include=["object", "string"]).columns if c != "merge_date"}
)

# SQL Server-də sətir sayını yoxlayırıq
with engine.connect() as conn:
    result = pd.read_sql(
        text("""
            SELECT COUNT(*) AS row_count
            FROM bronze.loans_enriched
        """),
        conn
    )

display(result)

,row_count
0,4820


Tranzaksiya məlumatlarını yoxlayaq

In [27]:
with engine.connect() as conn:
    transactions_df = pd.read_sql(
        text("SELECT * FROM bronze.transactions"),
        conn
    )
transactions_df.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel
0,T0000001,C02755,1/1/2026,7:03:37,21.05,AZN,Nağdlaşdırma,ATM
1,T0000002,C01051,1/1/2026,7:11:06,23.04,AZN,Nağdlaşdırma,ATM
2,T0000003,C03032,1/1/2026,7:11:29,54.70,AZN,Köçürmə,Mobil tətbiq
3,T0000004,C04099,1/1/2026,7:19:02,18.16,AZN,Mobil ödəniş,Mobil tətbiq
4,T0000005,C02686,1/1/2026,7:19:11,70.54,AZN,Nağdlaşdırma,ATM


Transactions + Customers

In [28]:
# Tranzaksiyaları müştəri məlumatları ilə birləşdiririk
bronze_transactions_merged = transactions_df.merge(
    customers_df,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

bronze_transactions_merged.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,city,branch,tenure_years,credit_score,balance,salary,products_count,has_card,is_active,churned
0,T0000001,C02755,1/1/2026,7:03:37,21.05,AZN,Nağdlaşdırma,ATM,67,Kişi,Bakı,Yasamal,11,653,3927.47,1568.44,1,0,1,1
1,T0000002,C01051,1/1/2026,7:11:06,23.04,AZN,Nağdlaşdırma,ATM,41,Qadın,Gəncə,Gəncə Mərkəz,7,647,3409.53,1133.34,1,0,1,1
2,T0000003,C03032,1/1/2026,7:11:29,54.70,AZN,Köçürmə,Mobil tətbiq,36,Qadın,Bakı,Binəqədi,5,783,0.00,973.82,2,1,1,0
3,T0000004,C04099,1/1/2026,7:19:02,18.16,AZN,Mobil ödəniş,Mobil tətbiq,25,Kişi,Bakı,Yasamal,4,696,5133.15,1278.68,2,1,1,0
4,T0000005,C02686,1/1/2026,7:19:11,70.54,AZN,Nağdlaşdırma,ATM,44,Kişi,Lənkəran,Lənkəran Mərkəz,6,705,1443.03,2168.40,1,1,1,0


Transactions+ Calendar
Tranzaksiya ile Təqvim məlumatlarını birləşdirək
İndi tranzaksiyalara bayram, həftəsonu və iş günü məlumatlarını əlavə edəcəyik.

In [29]:
# Tranzaksiya tarixini birləşdirməyə hazırlayırıq
bronze_transactions_merged["merge_date"] = pd.to_datetime(
    bronze_transactions_merged["date"]
).dt.date

# Təqvim məlumatlarını birləşdiririk
bronze_transactions_merged = bronze_transactions_merged.merge(
    calendar_df[
        [
            "merge_date",
            "is_holiday",
            "holiday_name",
            "is_weekend",
            "is_workday",
            "is_pre_holiday"
        ]
    ],
    on="merge_date",
    how="left",
    validate="many_to_one"
)
bronze_transactions_merged.head()


,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,...,products_count,has_card,is_active,churned,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday
0,T0000001,C02755,1/1/2026,7:03:37,21.05,AZN,Nağdlaşdırma,ATM,67,Kişi,...,1,0,1,1,2026-01-01,1,Yeni il,0,0,0
1,T0000002,C01051,1/1/2026,7:11:06,23.04,AZN,Nağdlaşdırma,ATM,41,Qadın,...,1,0,1,1,2026-01-01,1,Yeni il,0,0,0
2,T0000003,C03032,1/1/2026,7:11:29,54.70,AZN,Köçürmə,Mobil tətbiq,36,Qadın,...,2,1,1,0,2026-01-01,1,Yeni il,0,0,0
3,T0000004,C04099,1/1/2026,7:19:02,18.16,AZN,Mobil ödəniş,Mobil tətbiq,25,Kişi,...,2,1,1,0,2026-01-01,1,Yeni il,0,0,0
4,T0000005,C02686,1/1/2026,7:19:11,70.54,AZN,Nağdlaşdırma,ATM,44,Kişi,...,1,1,1,0,2026-01-01,1,Yeni il,0,0,0


Valyuta cədvəlini yoxlayaq
Bizdə bronze.exchange_rates cədvəlində AZN-dən USD və EUR-a məzənnələr var. Tranzaksiyalara bunları tarix və valyuta üzrə qoşacağıq.Amma əvvəlcə sütunları və valyutaları yoxlayaq.

In [30]:
with engine.connect() as conn:
    exchange_df = pd.read_sql(
        text("SELECT * FROM bronze.exchange_rates"),
        conn
    )
exchange_df.head()

,date,base,quote,rate
0,2026-01-01,AZN,EUR,0.50084
1,2026-01-01,AZN,USD,0.58837
2,2026-01-02,AZN,EUR,0.50095
3,2026-01-02,AZN,USD,0.58830
4,2026-01-03,AZN,EUR,0.50107


İndi məzənnələri tranzaksiyalara tarix və valyuta üzrə birləşdirəcəyik.

In [31]:
# Məzənnə tarixini uyğunlaşdırırıq
exchange_df["merge_date"] = pd.to_datetime(
    exchange_df["date"]
).dt.date

# Birləşdirmə üçün ayrıca cədvəl hazırlayırıq
rates_for_merge = exchange_df[
    ["merge_date", "quote", "base", "rate"]
].copy()

rates_for_merge = rates_for_merge.rename(
    columns={
        "quote": "rate_currency",
        "base": "rate_base"
    }
)

# Tranzaksiyalarla məzənnələri birləşdiririk
bronze_transactions_merged = bronze_transactions_merged.merge(
    rates_for_merge,
    left_on=["merge_date", "currency"],
    right_on=["merge_date", "rate_currency"],
    how="left",
    validate="many_to_one"
)

bronze_transactions_merged.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,...,churned,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday,rate_currency,rate_base,rate
0,T0000001,C02755,1/1/2026,7:03:37,21.05,AZN,Nağdlaşdırma,ATM,67,Kişi,...,1,2026-01-01,1,Yeni il,0,0,0,NaN,NaN,NaN
1,T0000002,C01051,1/1/2026,7:11:06,23.04,AZN,Nağdlaşdırma,ATM,41,Qadın,...,1,2026-01-01,1,Yeni il,0,0,0,NaN,NaN,NaN
2,T0000003,C03032,1/1/2026,7:11:29,54.70,AZN,Köçürmə,Mobil tətbiq,36,Qadın,...,0,2026-01-01,1,Yeni il,0,0,0,NaN,NaN,NaN
3,T0000004,C04099,1/1/2026,7:19:02,18.16,AZN,Mobil ödəniş,Mobil tətbiq,25,Kişi,...,0,2026-01-01,1,Yeni il,0,0,0,NaN,NaN,NaN
4,T0000005,C02686,1/1/2026,7:19:11,70.54,AZN,Nağdlaşdırma,ATM,44,Kişi,...,0,2026-01-01,1,Yeni il,0,0,0,NaN,NaN,NaN


Hava məlumatlarını tranzaksiyalara əlavə edək

In [32]:
# Birləşdirmədən əvvəl sətir sayı
before_count = len(bronze_transactions_merged)

# Hava məlumatlarını əlavə edirik
bronze_transactions_merged = bronze_transactions_merged.merge(
    weather_df[
        [
            "merge_date",
            "city",
            "temperature_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ]
    ],
    on=["merge_date", "city"],
    how="left",
    validate="many_to_one"
)

bronze_transactions_merged.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,...,holiday_name,is_weekend,is_workday,is_pre_holiday,rate_currency,rate_base,rate,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,T0000001,C02755,1/1/2026,7:03:37,21.05,AZN,Nağdlaşdırma,ATM,67,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
1,T0000002,C01051,1/1/2026,7:11:06,23.04,AZN,Nağdlaşdırma,ATM,41,Qadın,...,Yeni il,0,0,0,NaN,NaN,NaN,3.5,0.0,14.8
2,T0000003,C03032,1/1/2026,7:11:29,54.70,AZN,Köçürmə,Mobil tətbiq,36,Qadın,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
3,T0000004,C04099,1/1/2026,7:19:02,18.16,AZN,Mobil ödəniş,Mobil tətbiq,25,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
4,T0000005,C02686,1/1/2026,7:19:11,70.54,AZN,Nağdlaşdırma,ATM,44,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,7.2,0.3,10.8


Birləşdirilmiş tranzaksiyaları Bronze-a yazaq

In [33]:
# Birləşdirilmiş tranzaksiyaları SQL Server-ə yazırıq
bronze_transactions_merged.to_sql(
    name="transactions_enriched",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False,
dtype={c: Unicode(255) for c in bronze_transactions_merged.select_dtypes(include=["object", "string"]).columns if c != "merge_date"}
)



46

In [34]:
with engine.connect() as conn:
    transactions_enriched_df = pd.read_sql(
        text("SELECT * FROM bronze.transactions_enriched"),
        conn
    )
transactions_enriched_df.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,...,holiday_name,is_weekend,is_workday,is_pre_holiday,rate_currency,rate_base,rate,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,T0000001,C02755,1/1/2026,7:03:37,21.05,AZN,Nağdlaşdırma,ATM,67,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
1,T0000002,C01051,1/1/2026,7:11:06,23.04,AZN,Nağdlaşdırma,ATM,41,Qadın,...,Yeni il,0,0,0,NaN,NaN,NaN,3.5,0.0,14.8
2,T0000003,C03032,1/1/2026,7:11:29,54.70,AZN,Köçürmə,Mobil tətbiq,36,Qadın,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
3,T0000004,C04099,1/1/2026,7:19:02,18.16,AZN,Mobil ödəniş,Mobil tətbiq,25,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
4,T0000005,C02686,1/1/2026,7:19:11,70.54,AZN,Nağdlaşdırma,ATM,44,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,7.2,0.3,10.8


In [35]:
with engine.connect() as conn:
    loans_enriched_df = pd.read_sql(
        text("SELECT * FROM bronze.loans_enriched"),
        conn
    )
loans_enriched_df.head()

,loan_id,customer_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted,...,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday,time,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,L000001,C03267,1/1/2026,Biznes krediti,18174.08,34,11.34,Təsdiqləndi,NaN,0.0,...,2026-01-01,1,Yeni il,0,0,0,2026-01-01,5.5,0.0,18.6
1,L000002,C00082,1/2/2026,İstehlak krediti,14942.94,15,13.74,Rədd edildi,Yüksək borc yükü,NaN,...,2026-01-02,1,Yeni il,0,0,0,2026-01-02,5.2,12.9,22.6
2,L000003,C00715,1/2/2026,İstehlak krediti,24217.34,58,14.09,Təsdiqləndi,NaN,0.0,...,2026-01-02,1,Yeni il,0,0,0,2026-01-02,2.9,5.3,36.6
3,L000004,C02969,1/2/2026,Biznes krediti,14890.76,58,14.82,Rədd edildi,Aşağı kredit skoru,NaN,...,2026-01-02,1,Yeni il,0,0,0,2026-01-02,2.0,0.7,25.5
4,L000005,C00178,1/3/2026,İstehlak krediti,6382.58,21,13.19,Təsdiqləndi,NaN,0.0,...,2026-01-03,0,NaN,1,0,0,2026-01-03,3.6,0.0,10.7


Bronze cədvəllərini yoxlayaq

In [36]:
with engine.connect() as conn:
    bronze_tables = pd.read_sql(
        text("""
            SELECT TABLE_NAME
            FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = 'bronze'
            ORDER BY TABLE_NAME
        """),
        conn
    )

display(bronze_tables)

,TABLE_NAME
0,calendar_reference
1,customers
2,exchange_rates
3,loans
4,loans_enriched
5,transactions
6,transactions_enriched
7,weather
